# 案例3：通过特征工程提高模型的性能，以Wine数据集为例

本案例将通过Wine数据集来考察不同特征选择方法对模型性能的影响。<br>
Wine数据集来自 UCI 机器学习库的公开数据集，也是 scikit-learn 库的自带数据集。 它是对意大利同一地区种植的葡萄酒进行化学分析的结果，含有13种成分的数值，包括：
- 酒精(alcohol)
- 苹果酸(malic_acid)
- 灰(ash)
- 灰的碱性(alcalinity_of_ash)
- 镁 (magnesium)
- 总酚 (total_phenols)
- 类黄酮 (flavanoids)
- 非黄烷类酚类(nonflavanoid_phenols)
- 花青素(proanthocyanins)
- 颜色强度(color_intensity)
- 色调(hue)
- od280/od315
- 脯氨酸(proline)

这些葡萄酒来自三个不同的品种，共计 178条数据。


#### 安装必要的包

In [ ]:
# !pip install mrmr-selection==0.2.8


^C


   ---------------------------------------- 0.0/824.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/824.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/824.0 kB ? eta -:--:--
   ------------ --------------------------- 262.1/824.0 kB ? eta -:--:--
   ------------------------ ------------- 524.3/824.0 kB 901.1 kB/s eta 0:00:01
   ---------------------------------------- 824.0/824.0 kB 1.2 MB/s  0:00:01
   ---------------------------------------- 0.0/47.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/47.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/47.0 MB ? eta -:--:--
   ---------------------------------------- 0.5/47.0 MB 661.5 kB/s eta 0:01:11
   ---------------------------------------- 0.5/47.0 MB 661.5 kB/s eta 0:01:11
    --------------------------------------- 0.8/47.0 MB 652.6 kB/s eta 0:01:11
    --------------------------------------- 0.8/47.0 MB 652.6 kB/s eta 0:01:11
    ---------------------------

  Using cached mrmr_selection-0.2.8-py3-none-any.whl.metadata (6.6 kB)
  Using cached category_encoders-2.9.0-py3-none-any.whl.metadata (7.9 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached polars-1.39.3-py3-none-any.whl.metadata (10 kB)
  Using cached polars_runtime_32-1.39.3-cp310-abi3-win_amd64.whl.metadata (1.5 kB)
  Using cached patsy-1.0.2-py2.py3-none-any.whl.metadata (3.6 kB)
  Using cached statsmodels-0.14.6-cp313-cp313-win_amd64.whl.metadata (9.8 kB)
  Using cached markupsafe-3.0.3-cp313-cp313-win_amd64.whl.metadata (2.8 kB)
Using cached mrmr_selection-0.2.8-py3-none-any.whl (15 kB)
Using cached polars-1.39.3-py3-none-any.whl (823 kB)
   ---------------------------------------- 0.0/47.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/47.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/47.0 MB ? eta -:--:--
   ----------------------------------------

## 装载数据集

In [1]:
from sklearn.datasets import load_wine # 导入红酒数据集
from sklearn.model_selection import train_test_split # 划分数据集的模块
from sklearn.feature_selection import VarianceThreshold, chi2, SelectKBest
from sklearn.feature_selection import mutual_info_classif, f_classif
from mrmr import mrmr_classif
from sklearn.tree import DecisionTreeClassifier as DTC
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
import pandas as pd
# 1. 获得数据
wine = load_wine()
X, Y = wine.data, wine.target
num_class=6   #特征子集的大小
print(f"数据类型：{type(X)}")
print(f"数据样例个数：{X.shape[0]}；特征数目：{X.shape[1]}")


数据类型：<class 'numpy.ndarray'>
数据样例个数：178；特征数目：13


## 建立分类模型
本案例只是强调特征选择对模型性能的影响。不考虑何种模型最佳或如何达到最优模型。

下面定义了一个基于决策树的分类模型，并显示了不进行特征选择时，模型能够达到的Accuracy值。

In [2]:
def ClassifyingModel(X, Y):
    # 分割数据集
    X_train, X_test, y_train, y_test = train_test_split(X, Y, random_state=9)
    tree = DTC(max_depth=5, random_state=9) # 决策树模型
    tree.fit(X_train, y_train)
    score = tree.score(X_test, y_test, sample_weight=None) # 计算测试精度
    # knn = KNeighborsClassifier(n_neighbors=3)
    # knn.fit(X_train, y_train)
    # score = knn.score(X_test, y_test)
    return score

print("使用全部特征的模型准确度(x100%)： ", ClassifyingModel(X, Y))

使用全部特征的模型准确度(x100%)：  0.9555555555555556


## 1. 过滤法

下面用几种过滤法来进行特征选择，并评估特征选择的效果

### (1). 方差阈值法

本例子中设置方差阈值为1

In [5]:
vt_sel = VarianceThreshold(1) # 方差阈值法（阈值为 1）
vt_sel.fit(X)
vt_trans_X = vt_sel.transform(X)
print("方差阈值法选择的特征： ", vt_sel.get_support(True))
print("方差阈值法选择特征后的模型准确度（x100%）： ", ClassifyingModel(vt_trans_X, Y))

方差阈值法选择的特征：  [ 1  3  4  9 12]
方差阈值法选择特征后的模型准确度（x100%）：  0.8666666666666667


可以看到按照方差阈值法选择的特征反而使得模型降低了。大家可以自己试着更改方差阈值，观察选择的特征和模型的性能。

### (2). 卡方统计量法

In [6]:
chi_sel = SelectKBest(chi2, k=num_class)    # 卡方统计量法
chi_sel.fit(X, Y)
chi_trans_X = chi_sel.transform(X)
print("卡方统计量方法选择的特征： ", chi_sel.get_support(True))
print("卡方统计量法选择特征后的模型准确度（x100%）： ", ClassifyingModel(chi_trans_X, Y))

卡方统计量方法选择的特征：  [ 1  3  4  6  9 12]
卡方统计量法选择特征后的模型准确度（x100%）：  0.9555555555555556


### (3). 互信息法

In [31]:
mi_sel = SelectKBest(mutual_info_classif, k= num_class) # 互信息法
mi_sel.fit(X, Y)
mi_trans_X = mi_sel.transform(X)
print("互信息法选择的特征： ", mi_sel.get_support(True))
print("互信息法选择特征后的模型准确度（x100%）： ", ClassifyingModel(mi_trans_X, Y))

互信息法选择的特征：  [ 0  6  9 10 11 12]
互信息法选择特征后的模型准确度（x100%）：  0.9555555555555556


### (4). F统计量法

In [32]:
F_sel = SelectKBest(f_classif, k= num_class) # F 统计量法
F_sel.fit(X, Y)
F_trans_X = F_sel.transform(X)
print("F统计量法选择的特征： ", F_sel.get_support(True))
print("F 统计量法选择特征后的模型准确度（x100%）： ", ClassifyingModel(F_trans_X, Y))

F统计量法选择的特征：  [ 0  6  9 10 11 12]
F 统计量法选择特征后的模型准确度（x100%）：  0.9555555555555556


### (5). mRMR方法

In [36]:
X_df= pd.DataFrame(X, columns=wine['feature_names'] )
feature_index = mrmr_classif(X=X_df, y=Y,K=num_class) #mRMR方法
print("mRMR 方法选择的前几个特征名字： ", feature_index)  # 按照重要性降序

selected_indices = [X_df.columns.get_loc(f) for f in feature_index]
print("mRMR 方法选择的前几个特征序号： ", selected_indices)  # 按照重要性降序

mrmr_X=  X_df[feature_index]
print("mRMR 法选取特征后训练模型的准确度（x100%）： ", ClassifyingModel(mrmr_X, Y))

100%|████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:05<00:00,  1.10it/s]

mRMR 方法选择的前几个特征名字：  ['flavanoids', 'color_intensity', 'proline', 'od280/od315_of_diluted_wines', 'alcohol', 'hue']
mRMR 方法选择的前几个特征序号：  [6, 9, 12, 11, 0, 10]
mRMR 法选取特征后训练模型的准确度（x100%）：  0.9555555555555556


#### 总结过滤法

过滤式方法是一类常用的特征选择技术，其优缺点均非常明显。

优点: 算法的通用性强，省去了模型训练的步骤，算法复杂度低，因而适用于大规模数据集；可以快速去除大量不相关的特征，当原始数据的特征数量比较多时，作为特征的预筛选器非常合适。

缺点：由于特征选择过程独立于数据挖掘算法，所选择的特征子集对于数据挖掘任务而言通常不是最优的，性能经常低于其它两类方法

## 2. 包装法

利用 RFE 和 RFECV 类在 Wine 数据集上实现递归特征消除的特征选择

### (1) RFE 特征选择结果

In [38]:
from sklearn.feature_selection import RFE, RFECV
wine = load_wine()
X,Y= wine.data, wine.target
X_train, X_test, y_train, y_test = train_test_split(X, Y, random_state=9)

tree =DTC(criterion="entropy",max_depth=3, random_state=9)# 决策树模型

RFE_selector = RFE(estimator = tree, n_features_to_select = 5, step = 1)
RFE_selector.fit(X_train, y_train) # 在训练集上训练
print("RFE 选择的特征", RFE_selector.get_support(True))
print("RFE 方法选取特征所获得的测试精度",RFE_selector.score(X_test,y_test))


RFE 选择的特征 [ 6  9 10 11 12]
RFE 方法选取特征所获得的测试精度 0.9777777777777777


### (2) RFECV 特征选择结果

In [39]:
RFECV_selector = RFECV(estimator = tree, cv = 5, step = 1)
RFECV_selector.fit(X_train, y_train)
print("RFECV 选择的特征", RFECV_selector.get_support(True))
print("RFECV 方法选取特征所获得的测试精度",RFECV_selector.score(X_test,y_test))

RFECV 选择的特征 [ 6  9 10 12]
RFECV 方法选取特征所获得的测试精度 0.9777777777777777


可以观察到两个方法将特征选择方法和模型结合，都可以挑选优秀的特征来提高模型性能。

 ### (3) 利用 SequentialFeatureSelector 类在 Wine 数据集上实现序列特征选择

下面分别采用序列前向选择(SFS)和序列后向选择(SBS)进行特征选择

#### 首先建立一个评估函数

In [40]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.datasets import load_wine #导入红酒数据集
from sklearn.tree import DecisionTreeClassifier as DTC
from sklearn.model_selection import train_test_split # 划分数据集的模块
#辅助函数：特征子集的性能评价函数

def evaluate_select_subset(X_train, y_train, X_test, y_test, feature_index):
    Xtrain= X_train[:,feature_index]
    Xtest= X_test[:,feature_index]
    tree =DTC(criterion="entropy",max_depth=3, random_state=9)
    tree.fit(Xtrain,y_train)
    return tree.score(Xtest, y_test)


#### SFS 特征选择结果

In [41]:
# 1. 获得数据
wine = load_wine()
X,Y= wine.data, wine.target
X_train, X_test, y_train, y_test = train_test_split(X, Y, random_state=9)


tree =DTC(criterion="entropy",max_depth=3, random_state=9)# 决策树模型
SFS_selector = SequentialFeatureSelector(estimator = tree,
                                         n_features_to_select = 5, direction='forward')
SFS_selector.fit(X_train, y_train) # 训练
sd_feat= SFS_selector.get_support(True)
print("SFS 选择的特征", SFS_selector.get_support(True))
SFS_score= evaluate_select_subset(X_train, y_train, X_test, y_test, sd_feat)
print("SFS 选择的特征子集上获得的测试精度： ", SFS_score)


SFS 选择的特征 [ 0  6  9 10 12]
SFS 选择的特征子集上获得的测试精度：  0.9777777777777777


#### SBS 特征选择结果

In [42]:
tree =DTC(criterion="entropy",max_depth=3, random_state=9)# 决策树模型
SBS_selector = SequentialFeatureSelector(estimator = tree,
                                         n_features_to_select = 5, direction='backward')
SBS_selector.fit(X_train, y_train) # 训练
sd_feat= SBS_selector.get_support(True)
print("SBS 选择的特征", SBS_selector.get_support(True))
SBS_score= evaluate_select_subset(X_train, y_train, X_test, y_test, sd_feat)
print("SBS 选择的特征子集上获得的测试精度： ", SBS_score)

SBS 选择的特征 [ 6  8  9 10 12]
SBS 选择的特征子集上获得的测试精度：  0.9555555555555556


#### 总结包装法

包装法也是一类应用广泛的特征选择技术，其在选择选择过程考虑了数据挖掘模型的影响。它的主要优缺点包括：

优点：与过滤式特征选择方法相比，包装法的特征选择过程与数据挖掘任务相关，它使用后者的评价标准来对特征子集评分，使得选择结果是数据挖掘算法在其上表现最佳时的特征子集。并且，包装式方法对数挖掘模型没有过多要求，适用性比较广。

缺点：包装法是一种迭代式方法，对每一组特征子集都需要建立数据挖掘模型，在特征数量较多时，计算量非常大，效率远比过滤方法低。 另外， RFE、 SFS、 SBS 等包装法都采用启发式搜索方法寻找最优子集，它是一种局部搜索方法，因此这些方法搜索的最优子集可能是局部最优的。
